In [1]:
import re

In [2]:
import json

In [3]:
from dotenv import load_dotenv

In [4]:
load_dotenv()

True

In [5]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [6]:
loader = PyPDFLoader("DataEngineerStudyPlan.pdf")
pages = loader.load()

In [7]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

In [8]:
chunks = text_splitter.split_documents(pages)

In [9]:
text = ""

for i, chunk in enumerate(chunks):
    text += f"--- Чанк {i+1} ---\n"
    text += f"Метаданные (откуда кусок): {chunk.metadata}\n"
    text += f"Текст: {chunk.page_content}...\n"

print(text)

--- Чанк 1 ---
Метаданные (откуда кусок): {'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': 'DataEngineerStudyPlan.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}
Текст: Федеральное государственное автономное образовательное учреждение высшего образования "Национальный исследовательский университет 
"Высшая школа экономики"
Учебный план УТВЕРЖДЕН
27.05.2024
Направление подготовки 09.04.04 Программная инженерия Проректор
Образовательная программа "Инженерия данных" Рощин С.Ю.
Траектории: "Инженерия данных" Подписано ЭЦП
Реализующее подразделение: факультет компьютерных наук, Москва
Годы обучения: 2024/2025 учебный год - 2025/2026 учебный год
Срок обучения: 2 года
Форма обучения: очная
Уровень образования: Магистратура
№ п/п Наименование дисциплины Вид
дисциплины
Трудоемкость в 
зачетных единицах
Распределение 
зачетных единиц по 
годам обучения Планируемые результаты освоения 
образовательной программы
1 2
Вся образовательная программа 120,00 60,00 60,00
Инже

In [10]:
from transformers import AutoProcessor, AutoModelForCausalLM

In [11]:
MODEL_ID = "google/gemma-4-E2B-it"

In [12]:
processor = AutoProcessor.from_pretrained(MODEL_ID)

In [13]:
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype="auto", device_map="auto")

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

In [14]:
SYSTEM_PROMPT = """
You are a helpful assistant for processing study plans, extracted from pdf.
You will be given chunks of text from a study plan, and your task is to extract the key disciplines and their details.
Then create a bio for a potential student of this study plan, describe their interests and goals of this student.
Along with bio return a set of tags or interests, based on disciplines names.
Return the result strictly in JSON format with three fields: 'name', 'bio' and 'tags'.
For example:
{'name': 'machine_learning_enthusiast', 'bio': 'I am a student interested in machine learning and data science.', 'tags': ['machine_learning', 'data_science']}
"""

In [15]:
message = (
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": text},
)

In [16]:
def extract_json_from_text(text):
    match = re.search(r"```(?:json)?\s*([\s\S]*?)```", text)
    if match:
        return match.group(1).strip()
    match = re.search(r"\{[\s\S]*\}", text)
    if match:
        return match.group(0).strip()
    return text

In [17]:
text = processor.apply_chat_template(
    message, tokenize=False, add_generation_prompt=True, enable_thinking=False
)

inputs = processor(text=text, return_tensors="pt").to(model.device)
input_len = inputs["input_ids"].shape[-1]

outputs = model.generate(**inputs, max_new_tokens=1024)
response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)

json_str = extract_json_from_text(response)
answer = json.loads(json_str)

In [18]:
for k, v in answer.items():
    print(f"{k}: {v}")

name: DataEngineer_Master
bio: This student is highly motivated to pursue a Master's degree in Data Engineering, focusing on applying advanced computational and programming skills to large-scale data problems. They possess a strong interest in the entire data lifecycle, from data acquisition and processing (ETL) to database management, data visualization (BI analytics), and system architecture (DataOps). Their goal is to become a proficient Data Engineer capable of designing, implementing, and managing robust data pipelines and solutions for business needs.
tags: ['Data Engineering', 'Big Data', 'ETL', 'SQL', 'Python', 'Java', 'Database Management', 'DataOps', 'BI Analytics', 'Software Engineering', 'Algorithmics', 'System Design', 'Statistical Methods']
